### Generating names with recurrent neural networks

This time you'll find yourself delving into the heart (and other intestines) of recurrent neural networks on a class of toy problems.

Struggle to find a name for the variable? Let's see how you'll come up with a name for your son/daughter. Surely no human has expertize over what is a good child name, so let us train RNN instead;

It's dangerous to go alone, take these:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Our data
The dataset contains ~8k earthling names from different cultures, all in latin transcript.

This notebook has been designed so as to allow you to quickly swap names for something similar: deep learning article titles, IKEA furniture, pokemon names, etc.

In [ ]:
import os
start_token = " "

with open("names") as f:
    lines = f.read()[:-1].split('\n')
    lines = [start_token + name for name in lines]

In [ ]:
print('n samples = ', len(lines))
for x in lines[::1000]:
    print(x)

In [ ]:
MAX_LENGTH = max(map(len, lines))
print("max length =", MAX_LENGTH)

plt.title('Sequence length distribution')
plt.hist(list(map(len, lines)), bins=25)

# Text processing

First we need next to collect a "vocabulary" of all unique tokens i.e. unique characters. We can then encode inputs as a sequence of character ids.

In [ ]:
tokens = sorted(set("".join(lines)))

n_tokens = len(tokens)
print('n_tokens = ', n_tokens)

assert 50 < n_tokens < 60

### Cast everything from symbols into identifiers

Tensorflow string manipulation is a bit tricky, so we'll work around it. 
We'll feed our recurrent neural network with ids of characters from our dictionary.

To create such dictionary, let's assign each character with it's index in tokens list.

In [ ]:
# dictionary of symbol -> its identifier (index in tokens list)
token_to_id = {t: i for i, t in enumerate(tokens)}

In [ ]:
assert len(tokens) == len(token_to_id), "dictionaries must have same size"
for i in range(n_tokens):
    assert token_to_id[tokens[i]
                       ] == i, "token identifier must be it's position in tokens list"

print("Seems alright!")

In [ ]:
def to_matrix(names, max_len=None, pad=token_to_id[' '], dtype='int32'):
    """Casts a list of names into rnn-digestable matrix"""
    max_len = max_len or max(map(len, names))
    names_ix = np.zeros([len(names), max_len], dtype) + pad

    for i in range(len(names)):
        name_ix = list(map(token_to_id.get, names[i]))
        names_ix[i, :len(name_ix)] = name_ix

    return names_ix

In [ ]:
# Example: cast 4 random names to matrices, pad with zeros
print('\n'.join(lines[::2000]))
print(to_matrix(lines[::2000]))

# Recurrent neural network

We can rewrite recurrent neural network as a consecutive application of dense layer to input $x_t$ and previous rnn state $h_t$. This is exactly what we're gonna do now.
<img src="./rnn.png" width=480>

Since we're training a language model, there should also be:
* An embedding layer that converts character id x_t to a vector.
* An output layer that predicts probabilities of next token

In [ ]:
import os
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '-1')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import tensorflow as tf
from tensorflow import keras
import tensorflow.keras.layers as L

emb_size, rnn_size = 16, 64

embed_x = L.Embedding(n_tokens, emb_size)
get_h_next = L.Dense(rnn_size, activation='tanh')
get_probas = L.Dense(n_tokens, activation='softmax')

In [ ]:
def rnn_one_step(x_t, h_t):
    """One RNN step: takes x_t (int32[batch]) and h_t (float32[batch,rnn_size])
    and returns next hidden state and next-token probabilities."""
    x_t_emb = embed_x(tf.reshape(x_t, [-1, 1]))[:, 0]
    x_and_h = tf.concat([x_t_emb, h_t], axis=-1)
    next_h = get_h_next(x_and_h)
    next_probas = get_probas(next_h)
    return next_h, next_probas

In [ ]:
# In TF2 eager mode there are no placeholders; we build tensors on the fly.
def initial_h(batch_size):
    return tf.zeros([batch_size, rnn_size])

In [ ]:
dummy_data = tf.constant(np.arange(MAX_LENGTH * 2).reshape([2, -1]), dtype=tf.int32)
test_h1, test_p_y1 = rnn_one_step(dummy_data[:, 0], initial_h(2))
assert test_h1.shape == (2, rnn_size)
assert test_p_y1.shape == (2, n_tokens)
assert np.allclose(np.array(test_p_y1).sum(-1), 1)

### RNN loop

Once rnn_one_step is ready, let's apply it in a loop over name characters to get predictions.

Let's assume that all names are at most length-16 for now, so we can simply iterate over them in a for loop.




In [ ]:
def rnn_loop(input_sequence):
    h_prev = initial_h(tf.shape(input_sequence)[0])
    predicted = []
    for t in range(MAX_LENGTH):
        h_prev, probas_next = rnn_one_step(input_sequence[:, t], h_prev)
        predicted.append(probas_next)
    return tf.stack(predicted, axis=1), h_prev

In [ ]:
_test_seq = tf.constant(to_matrix(lines[:4], max_len=MAX_LENGTH), dtype=tf.int32)
_pred, _h = rnn_loop(_test_seq)
assert tuple(_pred.shape) == (4, MAX_LENGTH, n_tokens)
assert tuple(_h.shape) == (4, rnn_size)

## RNN: loss and gradients

Let's gather a matrix of predictions for $P(x_{next}|h)$ and the corresponding correct answers.

Our network can then be trained by minimizing crossentropy between predicted probabilities and those answers.

In [ ]:
# predictions vs answers are computed per-batch inside the train step below.
pass

In [ ]:
opt = keras.optimizers.Adam()


def trainable_vars():
    return embed_x.trainable_variables + get_h_next.trainable_variables + get_probas.trainable_variables


@tf.function
def train_step(input_sequence):
    with tf.GradientTape() as tape:
        predicted_probas, _ = rnn_loop(input_sequence)
        predictions_matrix = predicted_probas[:, :-1]
        answers_matrix = tf.one_hot(input_sequence[:, 1:], n_tokens)
        loss = -tf.reduce_mean(answers_matrix * tf.math.log(predictions_matrix + 1e-9))
    grads = tape.gradient(loss, trainable_vars())
    opt.apply_gradients(zip(grads, trainable_vars()))
    return loss

### The training loop

In [ ]:
from IPython.display import clear_output
from random import sample

history = []

In [ ]:
for i in range(1000):
    batch = tf.constant(to_matrix(sample(lines, 32), max_len=MAX_LENGTH), dtype=tf.int32)
    loss_i = train_step(batch)
    history.append(float(loss_i))
    if (i+1) % 200 == 0:
        print(f"iter {i+1:>4d}  loss = {np.mean(history[-50:]):.4f}")

plt.plot(history, label='loss'); plt.legend(); plt.show()
assert np.mean(history[:10]) > np.mean(history[-10:]), "RNN didn't converge."

### RNN: sampling
Once we've trained our network a bit, let's get to actually generating stuff. All we need is the `rnn_one_step` function you have written above.

In [ ]:
# In eager mode we'll just call rnn_one_step directly inside the sampling loop below.
pass

In [ ]:
def generate_sample(seed_phrase=' ', max_length=MAX_LENGTH):
    """Generate a name by sampling tokens one by one in eager mode."""
    x_sequence = [token_to_id[token] for token in seed_phrase]
    h = initial_h(1)
    for ix in x_sequence[:-1]:
        h, _ = rnn_one_step(tf.constant([ix], dtype=tf.int32), h)
    for _ in range(max_length - len(seed_phrase)):
        h, p = rnn_one_step(tf.constant([x_sequence[-1]], dtype=tf.int32), h)
        x_sequence.append(int(np.random.choice(n_tokens, p=np.array(p)[0])))
    return ''.join([tokens[ix] for ix in x_sequence])

In [ ]:
for _ in range(10):
    print(generate_sample())

In [ ]:
for _ in range(50):
    print(generate_sample(' Trump'))

### Try it out!
You've just implemented a recurrent language model that can be tasked with generating any kind of sequence, so there's plenty of data you can try it on:

* Novels/poems/songs of your favorite author
* News titles/clickbait titles
* Source code of Linux or Tensorflow
* Molecules in [smiles](https://en.wikipedia.org/wiki/Simplified_molecular-input_line-entry_system) format
* Melody in notes/chords format
* Ikea catalog titles
* Pokemon names
* Cards from Magic, the Gathering / Hearthstone

If you're willing to give it a try, here's what you wanna look at:
* Current data format is a sequence of lines, so a novel can be formatted as a list of sentences. Alternatively, you can change data preprocessing altogether.
* While some datasets are readily available, others can only be scraped from the web. Try `Selenium` or `Scrapy` for that.
* Make sure MAX_LENGTH is adjusted for longer datasets. There's also a bonus section about dynamic RNNs at the bottom.
* More complex tasks require larger RNN architecture, try more neurons or several layers. It would also require more training iterations.
* Long-term dependencies in music, novels or molecules are better handled with LSTM or GRU

__Good hunting!__

### Coming next

* The easy way to train recurrent neural networks in Keras
* Other problems solved with RNNs: sequence classification, sequential labelling
* LSTM, GRU, OMGWTF

```

```
```

```
```

```
```

```
```

```
```

```

### Bonus level: dynamic RNNs

Apart from keras, there's also a friendly tensorflow API for recurrent neural nets. It's based around the symbolic loop function (aka [scan](https://www.tensorflow.org/api_docs/python/tf/scan)).

This interface allows for dynamic sequence length and comes with some pre-implemented architectures.

In [ ]:
# TF2 equivalent: use a Keras RNN with a custom cell.
class CustomCell(L.AbstractRNNCell):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    @property
    def state_size(self):
        return rnn_size

    @property
    def output_size(self):
        return n_tokens

    def call(self, x_t, states):
        h = states[0]
        next_h, next_probas = rnn_one_step(x_t[:, 0], h)
        return next_probas, [next_h]


custom_rnn = L.RNN(CustomCell())
demo_in = tf.constant(to_matrix(lines[:10], max_len=50)[:, :, None], dtype=tf.int32)
demo_out = custom_rnn(demo_in)
print('CustomRNN output shape:', demo_out.shape)

Note that we never used MAX_LENGTH in the code above: TF will iterate over however many time-steps you gave it.

You can also use the all the pre-implemented RNN cells:

In [ ]:
# TF2 has Keras built-ins instead of tf.nn.rnn_cell / tf.contrib.rnn:
for name in dir(L):
    if name.endswith('Cell') or name in ('SimpleRNN', 'LSTM', 'GRU'):
        print(name)

In [ ]:
# Equivalent TF2/Keras LSTM block:
lstm_layer = L.LSTM(rnn_size, return_sequences=True)
demo_in = tf.constant(to_matrix(lines[:4], max_len=MAX_LENGTH), dtype=tf.int32)
state_sequence = lstm_layer(embed_x(demo_in))
print('LSTM visible states[batch, time, unit]:', state_sequence.shape)